# StormEngine — HadISD × ERA5: Full Data Aggregation & Analysis Pipeline

Complete four-step pipeline for the HadISD–ERA5 data aggregation chapter:

| Step | Content |
|---|---|
| **1** | Cross-temporal alignment — co-location table ERA5 × HadISD |
| **2** | Seasonal bias analysis — ERA5 vs real observations |
| **3** | HadISD data quality — missing rate, outliers, temporal consistency |
| **4** | Coverage assessment — spatial distribution & variable availability |

## Input files
| File | Content |
|---|---|
| `final_2024_11_msl.csv` | ERA5 MSL grid — November 2024 |
| `final_2024_12_msl.csv` | ERA5 MSL grid — December 2024 |
| `final_2024_1_msl.csv` | ERA5 MSL grid — January 2025 |
| `final_2024_2_msl.csv` | ERA5 MSL grid — February 2025 |
| `HadISD_Adriatic/*.csv` | HadISD point observations — same period |

## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats as sps
import glob, os, json, warnings, calendar
from pathlib import Path
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
})

# ── ERA5 monthly files ──────────────────────────────────────────────────────
# IMPORTANT FIX:
# The HadISD file used in this notebook is hadisd_adriatic_2024.csv, i.e. calendar year 2024.
# Therefore January and February must be timestamped as 2024, not 2025.
# If the intended experiment is winter 2024–2025, replace the HadISD file with one that includes 2025.
ERA5_FILES = {
    'January'  : {'path': 'final_2024_1_msl.csv',  'year': 2024, 'month':  1},
    'February' : {'path': 'final_2024_2_msl.csv',  'year': 2024, 'month':  2},
    'November' : {'path': 'final_2024_11_msl.csv', 'year': 2024, 'month': 11},
    'December' : {'path': 'final_2024_12_msl.csv', 'year': 2024, 'month': 12},
}
MONTH_ORDER = ['January', 'February', 'November', 'December']

# ── HadISD ──────────────────────────────────────────────────────────────────
HADISD_DIR  = 'hadisd_adriatic_2024.csv'   # raw/consolidated HadISD file, if available
HADISD_VAR  = 'PRESS'
COLOCATION_CSV = 'colocation_era5_hadisd_msl.csv'  # fallback / generated output

# ── Domain ──────────────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 39.0, 46.5
LON_MIN, LON_MAX = 12.0, 20.0
PA_TO_HPA        = 1.0 / 100.0

# ── Thresholds ───────────────────────────────────────────────────────────────
ALIGN_TOL_MIN   = 30
PHYS_MIN        = 950.0
PHYS_MAX        = 1060.0
MISSING_MAX_PCT = 30.0
JUMP_THRESHOLD  = 10.0

SEASON_MAP    = {11:'Winter', 2:'Winter', 11:'Autumn', 12:'Winter'}
# Expanded/explicit version to avoid ambiguity:
SEASON_MAP    = {1:'Winter', 2:'Winter', 11:'Autumn', 12:'Winter'}
MONTH_COLORS  = {'January':'#2c7fb8','February':'#41b6c4',
                 'November':'#fb8d3d','December':'#08519c'}
SEASON_COLORS = {'Winter':'#08519c','Autumn':'#fb8d3d'}

# ── Output folders ───────────────────────────────────────────────────────────
REPORT_DIR = Path('reports')
FIG_DIR = REPORT_DIR / 'figures' / 'hadisd_era5_full_pipeline'
TABLE_DIR = REPORT_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ── Verify all files exist before running ────────────────────────────────────
print('Working directory:', os.getcwd())
print()
all_era5_ok = True
for name, meta in ERA5_FILES.items():
    exists = os.path.isfile(meta['path'])
    print(f'  {"OK" if exists else "MISSING":6s} {meta["path"]}')
    if not exists:
        all_era5_ok = False

HAS_RAW_HADISD = os.path.isfile(HADISD_DIR)
HAS_EXISTING_COLOCATION = os.path.isfile(COLOCATION_CSV)
print(f'  {"OK" if HAS_RAW_HADISD else "MISSING":6s} {HADISD_DIR}')
if not HAS_RAW_HADISD and HAS_EXISTING_COLOCATION:
    print(f'  OK     fallback available: {COLOCATION_CSV}')

print()
if all_era5_ok and (HAS_RAW_HADISD or HAS_EXISTING_COLOCATION):
    print('Required inputs are available.')
else:
    print('WARNING: Some inputs are missing. The notebook can only run if ERA5 files and either raw HadISD or an existing co-location CSV are available.')


---
# STEP 1 — Cross-Temporal Alignment

Aligns HadISD point observations with ERA5 hourly gridded fields on the same timestamp (±30 min tolerance),
extracts ERA5 values at station coordinates via **bilinear interpolation**, and builds the
co-location table `(station_id, timestamp, hadisd_msl, era5_msl, bias)`.

## 1.1 Load ERA5 — Build Hourly Gridded Dictionary

In [ ]:
def load_era5_month(path, year, month):
    """Load one monthly ERA5 CSV exported from raster/grid format.

    Fixes included:
    - remove rows with invalid coordinates;
    - convert Pa → hPa only when the values look like Pa;
    - drop hourly columns that are entirely NaN;
    - keep coordinate/sample arrays aligned.
    """
    df = pd.read_csv(path)
    sc = [c for c in df.columns if c.startswith('SAMPLE_')]
    if not sc:
        raise ValueError(f'No SAMPLE_* columns found in {path}')
    if 'lon' not in df.columns or 'lat' not in df.columns:
        raise ValueError(f'Missing lon/lat columns in {path}')

    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    coord_ok = df['lat'].between(LAT_MIN, LAT_MAX) & df['lon'].between(LON_MIN, LON_MAX)
    df = df.loc[coord_ok].copy()

    base = pd.Timestamp(year=year, month=month, day=1, hour=0, tz='UTC')
    timestamps = [base + pd.Timedelta(hours=i) for i in range(len(sc))]

    samples = df[sc].apply(pd.to_numeric, errors='coerce').values.astype(float)
    if np.nanmedian(samples) > 2000:  # values are Pa
        samples = samples * PA_TO_HPA

    valid_time = ~np.all(np.isnan(samples), axis=0)
    samples = samples[:, valid_time]
    timestamps = [t for t, v in zip(timestamps, valid_time) if v]

    d = {
        'timestamps': timestamps,
        'samples'   : samples,
        'lon'       : df['lon'].values.astype(float),
        'lat'       : df['lat'].values.astype(float),
        'lons_uniq' : np.sort(df['lon'].dropna().unique().astype(float)),
        'lats_uniq' : np.sort(df['lat'].dropna().unique().astype(float)),
    }
    return d

era5 = {}
for month_name, meta in ERA5_FILES.items():
    try:
        d = load_era5_month(meta['path'], meta['year'], meta['month'])
        era5[month_name] = d
        nan_pct = np.isnan(d['samples']).mean() * 100
        print(f'{month_name:9s} | {d["timestamps"][0]} → {d["timestamps"][-1]} '
              f'| {len(d["timestamps"])} hours | {d["samples"].shape[0]} pts | NaN={nan_pct:.2f}%')
    except FileNotFoundError:
        print(f'{month_name}: NOT FOUND — {meta["path"]}')

# Diagnostic table for ERA5 quality
for month_name, d in era5.items():
    print(f'\n{month_name} ERA5 diagnostics')
    print('  samples shape:', d['samples'].shape)
    print('  NaN percentage:', round(np.isnan(d['samples']).mean()*100, 3))
    print('  min/max hPa:', round(np.nanmin(d['samples']), 3), round(np.nanmax(d['samples']), 3))


## 1.2 Load HadISD Observations

In [ ]:
def standardize_hadisd_like(df):
    """Standardize raw HadISD-like records to station_id, dt, value, lat, lon."""
    df = df.copy()
    if 'hadisd_msl' in df.columns and 'value' not in df.columns:
        df = df.rename(columns={'hadisd_msl': 'value'})
    if 'time' in df.columns and 'dt' not in df.columns:
        df = df.rename(columns={'time': 'dt'})
    if 'datetime' in df.columns and 'dt' not in df.columns:
        df = df.rename(columns={'datetime': 'dt'})
    if 'latitude' in df.columns and 'lat' not in df.columns:
        df = df.rename(columns={'latitude': 'lat'})
    if 'longitude' in df.columns and 'lon' not in df.columns:
        df = df.rename(columns={'longitude': 'lon'})

    required = ['station_id', 'dt', 'value', 'lat', 'lon']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required HadISD columns: {missing}. Existing columns: {list(df.columns)}')

    df['dt'] = pd.to_datetime(df['dt'], utc=True, errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

    if df['value'].median() > 2000:
        df['value'] = df['value'] * PA_TO_HPA

    df = df[required].dropna()
    df = df[df['lat'].between(LAT_MIN,LAT_MAX) & df['lon'].between(LON_MIN,LON_MAX)].copy()
    df['month_num']  = df['dt'].dt.month
    df['month_name'] = df['month_num'].map({1:'January',2:'February',11:'November',12:'December'})
    df['season']     = df['month_num'].map(SEASON_MAP)
    df = df[df['month_name'].notna()].copy()
    return df.sort_values('dt').reset_index(drop=True)


def load_hadisd(path_or_folder, var_code='PRESS'):
    if os.path.isfile(path_or_folder):
        files = [path_or_folder]
    else:
        files = sorted(glob.glob(os.path.join(path_or_folder, '*.csv')))
    if not files:
        raise FileNotFoundError(f'No CSV files found at: {path_or_folder}')

    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f))
        except Exception as e:
            print(f'  Warning: {f}: {e}')
    df = pd.concat(dfs, ignore_index=True)

    if 'sensor_code' in df.columns:
        df = df[df['sensor_code'] == var_code].copy()
    return standardize_hadisd_like(df)


def load_hadisd_from_colocation(path):
    """Fallback when raw hadisd_adriatic_2024.csv is not present.

    This reconstructs HadISD pressure records from the already matched co-location table.
    It is sufficient for report diagnostics, but it does not replace the full raw HadISD archive.
    """
    df = pd.read_csv(path)
    return standardize_hadisd_like(df)

if HAS_RAW_HADISD:
    hadisd = load_hadisd(HADISD_DIR, var_code=HADISD_VAR)
    HADISD_SOURCE = 'raw HadISD CSV'
elif HAS_EXISTING_COLOCATION:
    hadisd = load_hadisd_from_colocation(COLOCATION_CSV)
    HADISD_SOURCE = 'reconstructed from existing co-location CSV'
else:
    raise FileNotFoundError('Neither raw HadISD CSV nor existing co-location CSV is available.')

print(f'HadISD source    : {HADISD_SOURCE}')
print(f'HadISD obs       : {len(hadisd)}')
print(f'Unique stations  : {hadisd["station_id"].nunique()}')
print(f'Date range       : {hadisd["dt"].min()} → {hadisd["dt"].max()}')
print(f'Value range      : {hadisd["value"].min():.1f} — {hadisd["value"].max():.1f} hPa')
print(hadisd.groupby('month_name')[['station_id']].count().rename(columns={'station_id':'n_obs'}))


## 1.3 Bilinear Interpolation + Co-location Table

In [ ]:
def bilinear_interp_nan_safe(grid, lats, lons, qlat, qlon):
    """Bilinear interpolation with fallback for missing ERA5 cells.

    If the four surrounding grid cells are finite, use bilinear interpolation.
    If one or more surrounding values are NaN, use inverse-distance weighting from nearby finite cells.
    If no nearby finite cell exists, return NaN.
    """
    qlat = np.clip(qlat, lats[0], lats[-1])
    qlon = np.clip(qlon, lons[0], lons[-1])
    i1 = np.clip(np.searchsorted(lats, qlat)-1, 0, len(lats)-2)
    j1 = np.clip(np.searchsorted(lons, qlon)-1, 0, len(lons)-2)
    i2, j2 = i1+1, j1+1

    pts = np.array([
        [lats[i1], lons[j1], grid[i1, j1]],
        [lats[i2], lons[j1], grid[i2, j1]],
        [lats[i1], lons[j2], grid[i1, j2]],
        [lats[i2], lons[j2], grid[i2, j2]],
    ], dtype=float)
    valid = np.isfinite(pts[:, 2])

    if valid.all():
        lf = (qlat-lats[i1])/(lats[i2]-lats[i1]+1e-10)
        cf = (qlon-lons[j1])/(lons[j2]-lons[j1]+1e-10)
        return float(grid[i1,j1]*(1-lf)*(1-cf) + grid[i2,j1]*lf*(1-cf)
                    +grid[i1,j2]*(1-lf)*cf     + grid[i2,j2]*lf*cf)

    if valid.any():
        pts_v = pts[valid]
        dist = np.sqrt((pts_v[:,0]-qlat)**2 + (pts_v[:,1]-qlon)**2)
        if np.any(dist < 1e-12):
            return float(pts_v[np.argmin(dist), 2])
        w = 1 / (dist + 1e-12)
        return float(np.sum(w * pts_v[:, 2]) / np.sum(w))

    # Wider fallback: nearest finite cell in a 5x5 window around the target cell.
    i0 = int(np.clip(np.searchsorted(lats, qlat), 0, len(lats)-1))
    j0 = int(np.clip(np.searchsorted(lons, qlon), 0, len(lons)-1))
    i_min, i_max = max(0, i0-2), min(len(lats), i0+3)
    j_min, j_max = max(0, j0-2), min(len(lons), j0+3)
    window = grid[i_min:i_max, j_min:j_max]
    if np.isfinite(window).any():
        ii, jj = np.where(np.isfinite(window))
        lat_w = lats[i_min + ii]
        lon_w = lons[j_min + jj]
        vals = window[ii, jj]
        dist = np.sqrt((lat_w-qlat)**2 + (lon_w-qlon)**2)
        return float(vals[np.argmin(dist)])

    return np.nan


def reshape_grid(vals, lon, lat, lats_u, lons_u):
    g = np.full((len(lats_u), len(lons_u)), np.nan)
    li = {v:i for i,v in enumerate(lons_u)}
    la = {v:i for i,v in enumerate(lats_u)}
    for v,lo,lt in zip(vals,lon,lat):
        if not (np.isnan(lo) or np.isnan(lt)):
            g[la[lt],li[lo]] = v
    return g


def load_existing_colocation(path):
    df = pd.read_csv(path)
    # Standardize common column names.
    if 'time' in df.columns and 'dt' not in df.columns:
        df = df.rename(columns={'time':'dt'})
    if 'datetime' in df.columns and 'dt' not in df.columns:
        df = df.rename(columns={'datetime':'dt'})
    if 'latitude' in df.columns and 'lat' not in df.columns:
        df = df.rename(columns={'latitude':'lat'})
    if 'longitude' in df.columns and 'lon' not in df.columns:
        df = df.rename(columns={'longitude':'lon'})
    if 'value' in df.columns and 'hadisd_msl' not in df.columns:
        df = df.rename(columns={'value':'hadisd_msl'})

    required = ['station_id','dt','lat','lon','hadisd_msl','era5_msl']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Existing co-location table is missing columns: {missing}')

    df['dt'] = pd.to_datetime(df['dt'], utc=True, errors='coerce')
    if 'era5_dt' in df.columns:
        df['era5_dt'] = pd.to_datetime(df['era5_dt'], utc=True, errors='coerce')
    else:
        df['era5_dt'] = df['dt']
    for c in ['lat','lon','hadisd_msl','era5_msl']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    if df['era5_msl'].median() > 2000:
        df['era5_msl'] *= PA_TO_HPA
    if df['hadisd_msl'].median() > 2000:
        df['hadisd_msl'] *= PA_TO_HPA
    if 'bias' not in df.columns:
        df['bias'] = df['hadisd_msl'] - df['era5_msl']
    if 'dt_diff_min' not in df.columns:
        df['dt_diff_min'] = (df['dt'] - df['era5_dt']).abs().dt.total_seconds()/60
    df['month_num'] = df['dt'].dt.month
    if 'month_name' not in df.columns:
        df['month_name'] = df['month_num'].map({1:'January',2:'February',11:'November',12:'December'})
    if 'season' not in df.columns:
        df['season'] = df['month_num'].map(SEASON_MAP)
    df = df.dropna(subset=['station_id','dt','lat','lon','hadisd_msl','era5_msl','bias']).copy()
    return df.sort_values('dt').reset_index(drop=True)


def build_colocation(hadisd_df, era5_dict, tol_min=30):
    tol = pd.Timedelta(minutes=tol_min)
    rows, n_unmatched, n_nan_interp = [], 0, 0
    for month_name, d in era5_dict.items():
        era5_times = pd.DatetimeIndex(d['timestamps'])
        sub = hadisd_df[hadisd_df['month_name']==month_name].copy()
        if len(sub)==0:
            print(f'  {month_name}: no HadISD obs')
            continue
        print(f'  {month_name}: {len(sub)} obs → matching...')
        grids = {ts: reshape_grid(d['samples'][:,h], d['lon'], d['lat'],
                                  d['lats_uniq'], d['lons_uniq'])
                 for h, ts in enumerate(d['timestamps'])}
        for _, row in sub.iterrows():
            deltas = np.abs(era5_times - row['dt'])
            mi = int(deltas.argmin())
            if deltas[mi] > tol:
                n_unmatched += 1
                continue
            era5_val = bilinear_interp_nan_safe(grids[d['timestamps'][mi]],
                                                d['lats_uniq'], d['lons_uniq'],
                                                row['lat'], row['lon'])
            if not np.isfinite(era5_val):
                n_nan_interp += 1
                continue
            rows.append({'station_id':row['station_id'],'dt':row['dt'],
                         'era5_dt':d['timestamps'][mi],
                         'dt_diff_min':deltas[mi].total_seconds()/60,
                         'lat':row['lat'],'lon':row['lon'],
                         'month_name':month_name,'season':row['season'],
                         'hadisd_msl':row['value'],'era5_msl':era5_val,
                         'bias':row['value']-era5_val})
    coloc = pd.DataFrame(rows)
    print(f'\nMatched: {len(coloc)} | Unmatched by time: {n_unmatched} | Skipped NaN interpolation: {n_nan_interp}')
    return coloc

print('Building co-location table...')
if HAS_RAW_HADISD and era5:
    coloc = build_colocation(hadisd, era5, tol_min=ALIGN_TOL_MIN)
    coloc.to_csv(COLOCATION_CSV, index=False)
    print(f'Saved -> {COLOCATION_CSV}')
elif HAS_EXISTING_COLOCATION:
    print(f'Raw HadISD is unavailable; loading existing {COLOCATION_CSV} instead.')
    coloc = load_existing_colocation(COLOCATION_CSV)
else:
    raise RuntimeError('Cannot build or load co-location table.')

coloc.tail()


## 1.4 Alignment QC + Scatter Plot

In [ ]:
print(f'Total matched     : {len(coloc)}')
print(f'Unique stations   : {coloc["station_id"].nunique()}')
print(f'ERA5 range (hPa)  : {coloc["era5_msl"].min():.1f} — {coloc["era5_msl"].max():.1f}')
print(f'HadISD range (hPa): {coloc["hadisd_msl"].min():.1f} — {coloc["hadisd_msl"].max():.1f}')
print(f'NaN in era5_msl   : {coloc["era5_msl"].isna().sum()}')

# Temporal matching diagnostic
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(coloc['dt_diff_min'].dropna(), bins=30, color='steelblue', edgecolor='white')
ax.set(xlabel='Absolute time difference (minutes)', ylabel='Matched observations',
       title='Step 1 — Temporal Matching Quality')
plt.tight_layout(); plt.savefig(FIG_DIR / 's1_temporal_matching_quality.png', bbox_inches='tight'); plt.show()

months_p = [m for m in MONTH_ORDER if m in coloc['month_name'].values]
n = len(months_p)
fig, axes = plt.subplots(1, max(n,1), figsize=(5*max(n,1), 5), sharey=True, sharex=True)
if n == 1:
    axes = [axes]
if n == 0:
    print('No months available for scatter plot.')
else:
    for ax, month in zip(axes, months_p):
        sub = coloc[coloc['month_name']==month].dropna(subset=['era5_msl','hadisd_msl'])
        if sub.empty:
            ax.set_title(f'{month}\nno valid pairs')
            continue
        ax.scatter(sub['era5_msl'], sub['hadisd_msl'],
                   alpha=0.25, s=8, color=MONTH_COLORS.get(month, 'steelblue'))
        lims=[min(sub['era5_msl'].min(),sub['hadisd_msl'].min())-1,
              max(sub['era5_msl'].max(),sub['hadisd_msl'].max())+1]
        ax.plot(lims,lims,'k--',linewidth=1.2,label='1:1')
        bias = (sub['hadisd_msl']-sub['era5_msl']).mean()
        rmse = np.sqrt(((sub['hadisd_msl']-sub['era5_msl'])**2).mean())
        r    = sub[['era5_msl','hadisd_msl']].corr().iloc[0,1]
        ax.set_title(f'{month}\nbias={bias:+.2f}  RMSE={rmse:.2f}  r={r:.3f}')
        ax.set_xlabel('ERA5 MSL (hPa)'); ax.legend(fontsize=9)
    axes[0].set_ylabel('HadISD MSL (hPa)')
    fig.suptitle('Step 1 — ERA5 vs HadISD Scatter per Month', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.savefig(FIG_DIR / 's1_scatter.png', bbox_inches='tight'); plt.show()


---
# STEP 2 — Seasonal Bias Analysis

Quantifies the systematic discrepancy between ERA5 and real HadISD observations,
stratified by month and season. This is the methodological core of the Data Aggregation chapter.

## 2.1 Bias Statistics per Month and Season

In [ ]:
print('Bias summary (hPa) — HadISD minus ERA5:')
bias_stats = coloc.groupby('month_name')['bias'].agg(
    mean='mean', std='std', median='median',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
    n='count'
).round(3)
print(bias_stats.to_string())

print('\nSeasonal bias:')
print(coloc.groupby('season')['bias'].agg(
    mean='mean', std='std', median='median', n='count'
).round(3).to_string())

## 2.2 Boxplot Bias per Month and Season

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
month_order = [m for m in MONTH_ORDER if m in coloc['month_name'].values]
month_data = [coloc.loc[coloc['month_name']==m, 'bias'].dropna().values for m in month_order]
month_order = [m for m, vals in zip(month_order, month_data) if len(vals) > 0]
month_data = [vals for vals in month_data if len(vals) > 0]

if month_data:
    bp1 = ax1.boxplot(month_data, patch_artist=True, widths=0.55,
                      medianprops=dict(color='black',linewidth=1.5))
    for patch, m in zip(bp1['boxes'], month_order):
        patch.set_facecolor(MONTH_COLORS.get(m, 'gray')); patch.set_alpha(0.65)
    ax1.axhline(0,color='red',linestyle='--',linewidth=1)
    ax1.set_xticks(range(1,len(month_order)+1)); ax1.set_xticklabels(month_order)
else:
    ax1.text(0.5, 0.5, 'No valid monthly bias data', ha='center', va='center')
ax1.set_ylabel('Bias: HadISD − ERA5 (hPa)'); ax1.set_title('Bias per Month')

seasons = sorted([s for s in coloc['season'].dropna().unique()])
season_data = [coloc.loc[coloc['season']==s, 'bias'].dropna().values for s in seasons]
seasons = [s for s, vals in zip(seasons, season_data) if len(vals) > 0]
season_data = [vals for vals in season_data if len(vals) > 0]

if season_data:
    bp2 = ax2.boxplot(season_data, patch_artist=True, widths=0.45,
                      medianprops=dict(color='black',linewidth=1.5))
    for patch, s in zip(bp2['boxes'], seasons):
        patch.set_facecolor(SEASON_COLORS.get(s,'gray')); patch.set_alpha(0.65)
    ax2.axhline(0,color='red',linestyle='--',linewidth=1)
    ax2.set_xticks(range(1,len(seasons)+1)); ax2.set_xticklabels(seasons)
else:
    ax2.text(0.5, 0.5, 'No valid seasonal bias data', ha='center', va='center')
ax2.set_ylabel('Bias: HadISD − ERA5 (hPa)'); ax2.set_title('Bias per Season')
fig.suptitle('Step 2 — Seasonal Bias Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR / 's2_bias_boxplot.png', bbox_inches='tight'); plt.show()


## 2.3 Spatial Bias Map

In [ ]:
station_bias = coloc.groupby(['station_id','lat','lon'])['bias'].agg(
    mean_bias='mean', std_bias='std', n_obs='count'
).reset_index().dropna(subset=['mean_bias'])

vmax = station_bias['mean_bias'].abs().max()
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
if not station_bias.empty:
    sc1 = ax1.scatter(station_bias['lon'], station_bias['lat'],
                      c=station_bias['mean_bias'], cmap='RdBu_r', norm=norm,
                      s=station_bias['n_obs']/station_bias['n_obs'].max()*400+50,
                      edgecolors='black', linewidths=0.4, zorder=5)
    plt.colorbar(sc1, ax=ax1, label='Mean bias: HadISD − ERA5 (hPa)')
else:
    ax1.text(0.5, 0.5, 'No station bias data', ha='center', va='center')
ax1.set(xlim=(LON_MIN,LON_MAX), ylim=(LAT_MIN,LAT_MAX),
        xlabel='Lon (°E)', ylabel='Lat (°N)',
        title='Mean Bias per Station\n(size ∝ n observations)')

if not station_bias.empty:
    sc2 = ax2.scatter(station_bias['lon'], station_bias['lat'],
                      c=station_bias['std_bias'].fillna(0), cmap='viridis',
                      s=100, edgecolors='black', linewidths=0.4, zorder=5)
    plt.colorbar(sc2, ax=ax2, label='Bias std (hPa)')
else:
    ax2.text(0.5, 0.5, 'No station variability data', ha='center', va='center')
ax2.set(xlim=(LON_MIN,LON_MAX), ylim=(LAT_MIN,LAT_MAX),
        xlabel='Lon (°E)', ylabel='Lat (°N)', title='Bias Variability per Station')
fig.suptitle('Step 2 — Spatial Distribution of ERA5 Bias', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR / 's2_spatial_bias.png', bbox_inches='tight'); plt.show()

print('Top 5 most biased stations:')
if not station_bias.empty:
    print(station_bias.reindex(station_bias['mean_bias'].abs().sort_values(ascending=False).index)
          [['station_id','lat','lon','mean_bias','n_obs']].head(5).to_string(index=False))
else:
    print('No valid station bias values.')


## 2.4 Seasonal Bias per Station — Heatmap

In [ ]:
pivot = coloc.groupby(['station_id','season'])['bias'].mean().unstack(fill_value=np.nan)
if len(pivot) > 1 and np.isfinite(pivot.values).any():
    vmax = np.nanmax(np.abs(pivot.values))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    fig, ax = plt.subplots(figsize=(max(6,len(pivot.columns)*2+2), max(4,len(pivot)*0.4+2)))
    im = ax.imshow(pivot.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(s)[:16] for s in pivot.index], fontsize=8)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i,j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, label='Mean bias (hPa)')
    ax.set_title('Seasonal Bias per Station (hPa)')
    plt.tight_layout(); plt.savefig(FIG_DIR / 's2_bias_heatmap.png', bbox_inches='tight'); plt.show()
else:
    print('Not enough valid stations/seasons for heatmap — run with more stations.')


---
# STEP 3 — HadISD Data Quality

Although HadISD includes internal QC flags, an independent check is applied covering:
missing rate per station, physical plausibility, and temporal consistency (jump detection).
Stations exceeding quality thresholds are flagged for exclusion.

## 3.1 Missing Rate per Station

In [ ]:
# Expected observations: one per hour over the selected months.
# If HadISD was reconstructed from an existing co-location table, this missing-rate diagnostic
# refers to matched observations only, not the full raw archive.
total_hours = sum(
    calendar.monthrange(meta['year'], meta['month'])[1] * 24
    for meta in ERA5_FILES.values()
)
print(f'Expected hours in selected months: {total_hours}')

obs_per_station = hadisd.groupby('station_id').size().rename('n_obs')
missing_rate    = (1 - obs_per_station / total_hours) * 100
missing_rate    = missing_rate.clip(lower=0).rename('missing_pct')

station_qc = pd.DataFrame({'n_obs': obs_per_station, 'missing_pct': missing_rate})
station_qc['flagged_missing'] = station_qc['missing_pct'] > MISSING_MAX_PCT

print(f'\nStations total          : {len(station_qc)}')
print(f'Stations flagged (>{MISSING_MAX_PCT}% missing): {station_qc["flagged_missing"].sum()}')
print('\nMissing rate distribution:')
print(station_qc['missing_pct'].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['red' if f else 'steelblue' for f in station_qc['flagged_missing']]
ax.bar(range(len(station_qc)), station_qc['missing_pct'].values, color=colors, alpha=0.8)
ax.axhline(MISSING_MAX_PCT, color='red', linestyle='--', linewidth=1,
           label=f'Threshold ({MISSING_MAX_PCT}%)')
ax.set(xlabel='Station index', ylabel='Missing rate (%)',
       title='Step 3 — Missing Rate per HadISD Station')
ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 's3_missing_rate.png', bbox_inches='tight'); plt.show()


## 3.2 Physical Plausibility Check

In [ ]:
phys_bad = hadisd[(hadisd['value'] < PHYS_MIN) | (hadisd['value'] > PHYS_MAX)]
print(f'Physically implausible values (outside [{PHYS_MIN},{PHYS_MAX}] hPa): {len(phys_bad)}')
if len(phys_bad) > 0:
    print(phys_bad[['station_id','dt','value']].head(10).to_string(index=False))

# Z-score per station (flag individual anomalous readings)
hadisd_clean = hadisd.copy()
hadisd_clean['z_score'] = hadisd_clean.groupby('station_id')['value'].transform(
    lambda x: np.abs((x - x.mean()) / (x.std() + 1e-8))
)
z_outliers = hadisd_clean[hadisd_clean['z_score'] > 4]
print(f'Z-score > 4 outliers (per-station): {len(z_outliers)}')
if len(z_outliers) > 0:
    print(z_outliers[['station_id','dt','value','z_score']].head(10).to_string(index=False))

# Safer assignment: join by station_id instead of relying on array order.
phys_outliers_by_station = hadisd.groupby('station_id').apply(
    lambda x: ((x['value']<PHYS_MIN)|(x['value']>PHYS_MAX)).sum()
).rename('n_phys_outliers')
station_qc = station_qc.join(phys_outliers_by_station, how='left')
station_qc['n_phys_outliers'] = station_qc['n_phys_outliers'].fillna(0).astype(int)
station_qc['flagged_physical'] = station_qc['n_phys_outliers'] > 0

# Distribution per station
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(hadisd['value'], bins=60, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(PHYS_MIN, color='red', linestyle=':', label=f'Bounds [{PHYS_MIN},{PHYS_MAX}]')
ax.axvline(PHYS_MAX, color='red', linestyle=':')
ax.set(xlabel='MSL (hPa)', ylabel='Count',
       title='Step 3 — HadISD Value Distribution (all stations)')
ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 's3_distribution.png', bbox_inches='tight'); plt.show()


## 3.3 Temporal Consistency — Jump Detection

In [ ]:
def detect_jumps(df, threshold_hpa=10.0):
    """Flag consecutive-hour jumps exceeding threshold_hpa."""
    results = []
    for sid, grp in df.groupby('station_id'):
        grp = grp.sort_values('dt')
        diff = grp['value'].diff().abs()
        # Only flag jumps between consecutive hours
        dt_diff = grp['dt'].diff().dt.total_seconds() / 3600
        consec = (dt_diff <= 1.5)  # within 1.5h window
        n_jumps = ((diff > threshold_hpa) & consec).sum()
        results.append({'station_id': sid, 'n_jumps': int(n_jumps),
                        'max_jump': diff[consec].max() if consec.any() else 0})
    return pd.DataFrame(results)

jump_df = detect_jumps(hadisd, threshold_hpa=JUMP_THRESHOLD)
# Safer assignment: join by station_id instead of relying on array order.
jump_counts = jump_df.set_index('station_id')['n_jumps']
station_qc = station_qc.join(jump_counts, how='left')
station_qc['n_jumps'] = station_qc['n_jumps'].fillna(0).astype(int)
station_qc['flagged_jumps'] = station_qc['n_jumps'] > 0

print(f'Stations with temporal jumps > {JUMP_THRESHOLD} hPa: {station_qc["flagged_jumps"].sum()}')
print(jump_df[jump_df['n_jumps']>0].to_string(index=False))

# Plot time series of a station with most obs as sanity check
best_station = hadisd.groupby('station_id').size().idxmax()
ts = hadisd[hadisd['station_id']==best_station].sort_values('dt')
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ts['dt'], ts['value'], linewidth=0.8, color='steelblue')
ax.set(xlabel='Date', ylabel='MSL (hPa)',
       title=f'Step 3 — Temporal Consistency: {best_station}')
plt.tight_layout(); plt.savefig(FIG_DIR / 's3_temporal.png', bbox_inches='tight'); plt.show()


## 3.4 QC Summary — Flag and Report

In [ ]:
station_qc['flagged_any'] = (
    station_qc['flagged_missing'] | station_qc.get('flagged_jumps', False) | station_qc.get('flagged_physical', False)
)
print('='*55)
print('  STEP 3 — QC SUMMARY')
print('='*55)
print(f'  Total stations      : {len(station_qc)}')
print(f'  Flagged (missing)   : {station_qc["flagged_missing"].sum()}')
print(f'  Flagged (physical)  : {station_qc.get("flagged_physical", pd.Series(False, index=station_qc.index)).sum()}')
print(f'  Flagged (jumps)     : {station_qc.get("flagged_jumps", pd.Series(False, index=station_qc.index)).sum()}')
print(f'  Flagged (any)       : {station_qc["flagged_any"].sum()}')
print(f'  Clean stations      : {(~station_qc["flagged_any"]).sum()}')
print('='*55)
print(station_qc.to_string())

station_qc.to_csv(TABLE_DIR / 'hadisd_station_qc.csv', index=True)
station_qc.to_csv('hadisd_station_qc.csv', index=True)  # keep backward-compatible output
print(f'\nSaved -> {TABLE_DIR / "hadisd_station_qc.csv"}')


---
# STEP 4 — Coverage Assessment

Characterises the spatial distribution of HadISD stations over the domain,
their variable availability, and the zones of the ERA5 grid not covered by any station.

## 4.1 Station Map — HadISD vs ERA5 Grid

In [ ]:
# ERA5 grid points from first available month
if not era5:
    raise RuntimeError('No ERA5 data loaded; cannot plot station map.')
era5_sample = list(era5.values())[0]

fig, ax = plt.subplots(figsize=(10, 8))

# ERA5 grid (background)
ax.scatter(era5_sample['lon'], era5_sample['lat'],
           s=4, color='lightgray', alpha=0.6, label='ERA5 grid', zorder=1)

# HadISD stations coloured by QC status
sta_info = hadisd.groupby(['station_id','lat','lon']).size().reset_index(name='n_obs')
sta_info = sta_info.merge(
    station_qc[['flagged_any']].reset_index(), on='station_id', how='left'
)
clean = sta_info[~sta_info['flagged_any'].fillna(False)]
flagged = sta_info[sta_info['flagged_any'].fillna(False)]

if len(clean):
    ax.scatter(clean['lon'], clean['lat'],
               s=clean['n_obs']/clean['n_obs'].max()*200+40,
               color='steelblue', edgecolors='black', linewidths=0.5,
               zorder=5, label=f'HadISD — clean ({len(clean)})')
if len(flagged):
    ax.scatter(flagged['lon'], flagged['lat'],
               s=80, color='red', marker='x', linewidths=1.5,
               zorder=6, label=f'HadISD — flagged ({len(flagged)})')

# Annotate station IDs
for _, row in sta_info.iterrows():
    ax.annotate(str(row['station_id']).split('-')[0],
                (row['lon'], row['lat']), fontsize=6,
                xytext=(3, 3), textcoords='offset points', color='black')

ax.set(xlim=(LON_MIN-0.5, LON_MAX+0.5), ylim=(LAT_MIN-0.5, LAT_MAX+0.5),
       xlabel='Longitude (°E)', ylabel='Latitude (°N)',
       title='Step 4 — HadISD Stations vs ERA5 Grid\n(marker size ∝ n observations)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(FIG_DIR / 's4_station_map.png', bbox_inches='tight'); plt.show()


## 4.2 Variable Availability per Station

In [ ]:
# Load ALL sensor codes to check variable availability across stations.
# Fixed to work when HADISD_DIR is a file, not only a folder.
def load_hadisd_all_vars(path_or_folder):
    if os.path.isfile(path_or_folder):
        files = [path_or_folder]
    else:
        files = sorted(glob.glob(os.path.join(path_or_folder, '*.csv')))
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f))
        except Exception as e:
            print(f'Warning: could not read {f}: {e}')
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    if 'latitude' in df.columns and 'lat' not in df.columns:
        df = df.rename(columns={'latitude':'lat'})
    if 'longitude' in df.columns and 'lon' not in df.columns:
        df = df.rename(columns={'longitude':'lon'})
    if 'lat' in df.columns and 'lon' in df.columns:
        df = df[df['lat'].between(LAT_MIN,LAT_MAX) & df['lon'].between(LON_MIN,LON_MAX)]
    return df

df_all = load_hadisd_all_vars(HADISD_DIR) if HAS_RAW_HADISD else pd.DataFrame()

if 'sensor_code' in df_all.columns and len(df_all)>0:
    avail = df_all.groupby(['station_id','sensor_code']).size().unstack(fill_value=0)
    avail_bool = (avail > 0).astype(int)
    print('Variable availability per station:')
    print(avail_bool.to_string())

    fig, ax = plt.subplots(figsize=(max(6,len(avail_bool.columns)+2),
                                    max(4,len(avail_bool)*0.4+2)))
    im = ax.imshow(avail_bool.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(avail_bool.columns)))
    ax.set_xticklabels(avail_bool.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(avail_bool.index)))
    ax.set_yticklabels([str(s)[:18] for s in avail_bool.index], fontsize=8)
    for i in range(len(avail_bool.index)):
        for j in range(len(avail_bool.columns)):
            ax.text(j, i, '✓' if avail_bool.values[i,j] else '✗',
                    ha='center', va='center', fontsize=9,
                    color='white' if avail_bool.values[i,j] else 'lightgray')
    ax.set_title('Step 4 — Variable Availability per Station')
    plt.tight_layout(); plt.savefig(FIG_DIR / 's4_variable_availability.png', bbox_inches='tight'); plt.show()
else:
    print('Raw multi-variable HadISD file or sensor_code column not available — skipping variable availability heatmap.')


## 4.3 Uncovered ERA5 Grid Cells

In [ ]:
# For each ERA5 grid point, find distance to nearest HadISD station.
# Fixes included:
# - drop NaN coordinates before distance calculation;
# - use paired station lon/lat, not separate unique arrays;
# - prevent x/y length mismatch in scatter plot.
sta_coords_df = hadisd[['lat','lon']].drop_duplicates().dropna()
era5_pts_df = pd.DataFrame({
    'lat': era5_sample['lat'],
    'lon': era5_sample['lon'],
}).dropna()

sta_coords = sta_coords_df[['lat','lon']].values
era5_pts   = era5_pts_df[['lat','lon']].values

if len(sta_coords) == 0 or len(era5_pts) == 0:
    raise RuntimeError('No valid station or ERA5 coordinates available for coverage assessment.')

def min_distance_km(era5_pts, sta_coords):
    """Approximate great-circle distance in km using local flat-earth approximation."""
    dists = []
    for ep in era5_pts:
        d = np.sqrt(((sta_coords[:,0]-ep[0])*111)**2 +
                    ((sta_coords[:,1]-ep[1])*111*np.cos(np.radians(ep[0])))**2)
        dists.append(np.nanmin(d))
    return np.array(dists)

min_dist = min_distance_km(era5_pts, sta_coords)
threshold_km = 150  # consider uncovered if no station within 150 km

print(f'ERA5 grid points total           : {len(era5_pts)}')
print(f'Covered (within {threshold_km} km)       : {(min_dist<=threshold_km).sum()}')
print(f'Uncovered (> {threshold_km} km)          : {(min_dist>threshold_km).sum()}')
print(f'Mean distance to nearest station : {np.nanmean(min_dist):.1f} km')
print(f'Max distance to nearest station  : {np.nanmax(min_dist):.1f} km')

# Map
def to_grid_vec(vals, lon, lat, lats_u, lons_u):
    g = np.full((len(lats_u), len(lons_u)), np.nan)
    li={v:i for i,v in enumerate(lons_u)}; la={v:i for i,v in enumerate(lats_u)}
    for v,lo,lt in zip(vals,lon,lat):
        if not (np.isnan(lo) or np.isnan(lt)) and lo in li and lt in la:
            g[la[lt],li[lo]] = v
    return g

dist_grid = to_grid_vec(min_dist, era5_pts_df['lon'].values, era5_pts_df['lat'].values,
                        era5_sample['lats_uniq'], era5_sample['lons_uniq'])

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(dist_grid, cmap='YlOrRd',
               extent=[LON_MIN,LON_MAX,LAT_MIN,LAT_MAX], aspect='auto', origin='lower')
plt.colorbar(im, ax=ax, label='Distance to nearest HadISD station (km)')
ax.scatter(sta_coords_df['lon'], sta_coords_df['lat'],
           s=60, color='blue', edgecolors='black', linewidths=0.5,
           zorder=5, label='HadISD stations')
try:
    ax.contour(dist_grid, levels=[threshold_km], colors='red', linewidths=1.5,
               extent=[LON_MIN,LON_MAX,LAT_MIN,LAT_MAX], origin='lower')
except Exception as e:
    print(f'Coverage contour skipped: {e}')
ax.set(xlim=(LON_MIN,LON_MAX), ylim=(LAT_MIN,LAT_MAX),
       xlabel='Lon (°E)', ylabel='Lat (°N)',
       title=f'Step 4 — ERA5 Grid Coverage\n(red contour = {threshold_km} km threshold)')
ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 's4_coverage_map.png', bbox_inches='tight'); plt.show()


---
# Final Report Summary

In [ ]:
print('='*60)
print('  FULL PIPELINE SUMMARY')
print('='*60)
print(f'  STEP 1 — Alignment')
print(f'    Co-location pairs        : {len(coloc)}')
print(f'    Unique stations matched  : {coloc["station_id"].nunique()}')
print(f'    Overall mean bias (hPa)  : {coloc["bias"].mean():+.3f}')
print(f'    Overall RMSE  (hPa)      : {np.sqrt((coloc["bias"]**2).mean()):.3f}')
print()
print(f'  STEP 2 — Seasonal bias')
for s, grp in coloc.groupby('season'):
    print(f'    {s:7s}: mean={grp["bias"].mean():+.3f}  std={grp["bias"].std():.3f} hPa')
print()
print(f'  STEP 3 — HadISD quality')
print(f'    Total stations           : {len(station_qc)}')
print(f'    Flagged stations         : {station_qc["flagged_any"].sum()}')
print(f'    Clean stations           : {(~station_qc["flagged_any"]).sum()}')
print()
print(f'  STEP 4 — Coverage')
print(f'    ERA5 grid points         : {len(era5_pts)}')
print(f'    Mean dist to station (km): {np.nanmean(min_dist):.1f}')
print(f'    Uncovered pts (>{threshold_km}km)  : {(min_dist>threshold_km).sum()}')
print()
print('  Saved figure folder:')
print(f'    {FIG_DIR}')
print('  Saved table folder:')
print(f'    {TABLE_DIR}')
print('='*60)
